# Phase 5b: A/B 测试专项特训 (A/B Testing Specialization) 🧪

> **场景**: 你正在面试 某短视频大厂B公司/某新能源车企T公司 的 Senior DA 岗位。
> **目标**: 面试官想考察你在不同业务场景下的统计推断能力。不要只懂 T-Test，要懂业务。

---

## 💼 Case 1: 点击率 (Click-Through Rate) 实验
**难度**: ⭐⭐⭐ | **核心考点**: 数据清洗、分母校准、T-Test

### 1.1 背景 (Context)
- **产品变动**: 对 App 首页的 "猜你喜欢" 算法进行了升级 (Test组) vs 旧算法 (Control组)。
- **原始数据**: 给到你的是两张表：
    1.  `experiment_assignment.csv`: 用户被分到了哪一组。
    2.  `user_activity_logs.json` (模拟): 用户的点击行为日志，是一个嵌套的 JSON 字符串，非常脏。

### 1.2 任务 (Tasks)
1.  **Parsing**: 解析 JSON 格式的行为日志 (或提取关键字段)。
2.  **Cleaning & Joining**: 清洗数据，注意**保留未点击的用户 (Zero Handling)**。
3.  **Metrics Calculation**: 计算核心指标 —— **人均点击次数** 和 **点击率**。
4.  **Statistical Inference**: 运行 T-Test，判断差异是否显著。 
5.  **Decision**: 输出结论 (Go / No-Go)。

In [ ]:
import pandas as pd
import numpy as np
import json
import scipy.stats as stats

# ---------------- 0. Case 1 造数据 ----------------
np.random.seed(42)
user_ids = [f'U{str(i).zfill(4)}' for i in range(1000)]
groups = np.random.choice(['Control', 'Test'], size=1000)
df_assignment = pd.DataFrame({'user_id': user_ids, 'group': groups})

# 模拟：脏 JSON 日志
raw_logs = [
    '{"uid": "U0001", "evt": "view", "ts": "2024-01-01 10:00:00"}',
    '{"uid": "U0001", "evt": "click", "ts": "2024-01-01 10:00:05", "meta": {"btn": "like"}}',
    '{"uid": "U0002", "evt": "view", "ts": "2024-01-01 10:01:00"}',
    '{"uid": "U0005", "evt": "error", "msg": "timeout"}', 
    '{"uid": "U0003", "evt": "click", "ts": "2024-01-01 12:00:00"}'
] * 500
np.random.shuffle(raw_logs)
print("Case 1 Data Generated.")
# -------------------------------------------------

In [ ]:
# --- Case 1: 你的代码 ---
# 1.  **Parsing**: 解析 JSON 格式的行为日志 (或提取关键字段)。
raw_logs = [json.loads(line) for line in raw_logs]
raw_logs = pd.DataFrame(raw_logs)
raw_logs

In [ ]:
# 2.  **Cleaning & Joining**: 清洗数据，注意**保留未点击的用户 (Zero Handling)**。
raw_logs['user_id'] = raw_logs['uid']
df_assignment = df_assignment.merge(raw_logs,how='left',on='user_id')
df_assignment.describe()

In [ ]:
# 3.  **Metrics Calculation**: 计算核心指标 —— **人均点击次数** 和 **点击率**。
df_assignment['click_cnt'] = np.where(df_assignment['evt']=='click',1,0)
click_per_cnt = df_assignment.groupby('group')['click_cnt'].sum()/df_assignment.groupby('group')['user_id'].nunique()
click_per_cnt


In [ ]:
# 4.  **Statistical Inference**: 运行 T-Test，判断差异是否显著。 
import scipy.stats as stats 
control_group = df_assignment[df_assignment['group'] == 'Control']
test_group = df_assignment[df_assignment['group'] == 'Test']
stats,p_value = stats.ttest_ind(control_group['click_cnt'],test_group['click_cnt'])

In [ ]:
# 5.  **Decision**: 输出结论 (Go / No-Go)。
result = np.where(p_value<0.05,"Go","No-Go")
print(f"{result}")


## 💰 Case 2: 支付金额 A/B 实验 (ARPU Test)
**难度**: ⭐⭐⭐⭐ | **核心考点**: 连续值统计、0值填充、异常值处理

### 2.1 背景
- **问题**: 我们测试了新的会员价格页 (Test组) vs 旧版 (Control组)。老板不仅关心转化率，更关心 **ARPU (每用户平均收入)**。
- **数据**: `df_payments` (只有付费用户的记录)，以及 `df_assignment` (所有实验用户)。

### 2.2 任务
1.  **Metric Definition**: 计算 ARPU (总收入 / 总实验人数)。*注意区分 ARPU 和 ARPPU*。
2.  **Handling Zeros**: 很多用户根本没买 (Revenue=0)。如果不补0直接算均值，ARPU 会虚高。
3.  **Statistical Inference**: 针对这种"大量0 + 少量大额支付"的偏态分布，运行 T-Test。
4.  **Outlier Check**: 检查是否需要剔除极值 (Whales)。

In [ ]:
import pandas as pd
import numpy as np
# import json


# ---------------- 0. Case 2 造数据 ----------------
np.random.seed(2024)
# 2000个用户
users_c2 = [f'User_{i}' for i in range(2000)]
groups_c2 = np.random.choice(['Control', 'Test'], size=2000)
df_assigned_c2 = pd.DataFrame({'user_id': users_c2, 'group': groups_c2})

# 模拟付费：只有 5% 的人付费，金额服从指数分布 (长尾)
paying_users_idx = np.random.choice(range(2000), size=100, replace=False)
paying_users = [users_c2[i] for i in paying_users_idx]
revenue = np.random.exponential(scale=50, size=100).round(2) + 10 # 至少10块

# 制造一个 Test组的土豪 (Outlier)，扭曲实验结果
revenue[0] = 50000 

df_payments = pd.DataFrame({'user_id': paying_users, 'revenue': revenue})
print("Case 2 Data Generated.")
print(df_payments.head())
print(df_payments.info())
print(df_assigned_c2.head())
# -------------------------------------------------

In [ ]:
# --- Case 2: 你的代码 ---
# 1.  **Metric Definition**: 计算 ARPU (总收入 / 总实验人数)。*注意区分 ARPU 和 ARPPU*。
df_assigned_c2 = df_assigned_c2.merge(df_payments,how='left',on = 'user_id')
df_assigned_c2['revenue'] = df_assigned_c2['revenue'].fillna(0)
df_assigned_c2

In [ ]:
df_assigned_c2['control_amount'] = np.where(df_assigned_c2['group']=='Control',df_assigned_c2['revenue'],0)
df_assigned_c2['test_amount'] = np.where(df_assigned_c2['group']=='Test',df_assigned_c2['revenue'],0)
df_assigned_c2['control_users'] = np.where(df_assigned_c2['group']=='Control',1,0)
df_assigned_c2['test_users'] = np.where(df_assigned_c2['group']=='Test',1,0)
# df_assigned_c2.info()



In [ ]:
ARPU = df_assigned_c2.groupby('group')['revenue'].sum()/df_assigned_c2.groupby('group')['user_id'].nunique()
print(f"ARPU is {round(ARPU)}")

In [ ]:
# 2.  **Handling Zeros**: 很多用户根本没买 (Revenue=0)。如果不补0直接算均值，ARPU 会虚高。
# 3.  **Statistical Inference**: 针对这种"大量0 + 少量大额支付"的偏态分布，运行 T-Test。
import scipy.stats as stats
control_amount = df_assigned_c2[df_assigned_c2['group']=='Control']
test_amount = df_assigned_c2[df_assigned_c2['group']=='Test']
stats,p_value = stats.ttest_ind(df_assigned_c2['control_amount'],df_assigned_c2['test_amount'])
# 4.  **Outlier Check**: 检查是否需要剔除极值 (Whales)。
p_value


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 假设你已经把表 merge 好了，每一行有 'group' 和 'revenue'
# 如果还没 merge，记得先把 df_assignment 和 df_payments 连起来！

plt.figure(figsize=(6, 4))
# x轴放分组，y轴放金额
sns.boxplot(data=df_assigned_c2, x='group', y='revenue') 
plt.show()

In [ ]:
import scipy.stats as stats

df_clean = df_assigned_c2[df_assigned_c2['revenue'] <50000]
df_clean
control_amount = df_clean[df_clean['group']=='Control']
test_amount = df_clean[df_clean['group']=='Test']
stats,p_value = stats.ttest_ind(control_amount['revenue'],test_amount['revenue'])
print(f"p_value: {p_value}")

## 📉 Case 3: 全链路漏斗转化实验 (Funnel Test)
**难度**: ⭐⭐⭐⭐ | **核心考点**: 转化率分析、卡方检验 (Chi-Square)

### 3.1 背景
- **问题**: 产品经理简化了结账流程。我们要看从 "浏览(View)" -> "加购(Cart)" -> "支付(Pay)" -> "成功(Success)" 的每一步转化率是否有提升。
- **数据**: `df_funnel` 包含每个用户到达的 **最大步骤 (Max Step)**。

### 3.2 任务
1.  **Funnel Construction**: 计算各环节的**绝对人数**和**转化率**。
2.  **Hypothesis Testing**: 使用 **Chi-Square Test (卡方检验)** 来判断两个漏斗分布是否有显著差异。
    *   *提示*: `scipy.stats.chi2_contingency`

In [1]:
import pandas as pd
import numpy as np
# ---------------- 0. Case 3 造数据 ----------------
np.random.seed(99)
n = 3000
groups_c3 = np.random.choice(['Control', 'Test'], size=n)
# 模拟漏斗步骤: View -> Cart -> Pay -> Success
steps = ['View', 'Cart', 'Pay', 'Success']
# 随机生成用户到达的最大步骤 (Control组稍差，Test组稍好)
probs_control = [0.4, 0.3, 0.2, 0.1]
probs_test =    [0.35, 0.3, 0.2, 0.15] # Test组 Success 变高了

max_steps = []
for g in groups_c3:
    p = probs_control if g == 'Control' else probs_test
    max_steps.append(np.random.choice(steps, p=p))

df_funnel = pd.DataFrame({'user_id': range(n), 'group': groups_c3, 'max_step': max_steps})
print("Case 3 Data Generated.")
print(df_funnel.head())
# -------------------------------------------------

Case 3 Data Generated.
   user_id    group max_step
0        0     Test  Success
1        1     Test     Cart
2        2     Test     View
3        3  Control     View
4        4     Test      Pay


In [2]:
# --- 3.3 进阶漏斗分析 (Best Practice) ---
# 这段代码会自动对【相邻两个环节】做卡方检验

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

# 🔎 自动修复: 防止 NameError (如果忘了运行上面的 cell)
try:
    funnel
except NameError:
    print("⚠️ 检测到 funnel 未定义，正在从 pd_funnel 重新构建...")
    step_order = ['View', 'Cart', 'Pay', 'Success']
    df_funnel['max_step'] = pd.Categorical(df_funnel['max_step'], categories=step_order, ordered=True)
    funnel = df_funnel.groupby(['group','max_step'])['user_id'].nunique().unstack()

# 必要的修正：确保 group 是 index
if 'group' in funnel.columns:
    funnel = funnel.set_index('group')

# 定义漏斗步骤顺序
funnel_steps = ['View', 'Cart', 'Pay', 'Success']
results = []

# 循环每一个相邻的环节
for i in range(len(funnel_steps)-1):
    this_step = funnel_steps[i]   # 当前环节 (分母)
    next_step = funnel_steps[i+1] # 下一环节 (分子)
    
    # 构造该环节的 2x2 矩阵
    contingency = np.array([
        [funnel.loc['Control', next_step], funnel.loc['Control', this_step] - funnel.loc['Control', next_step]],
        [funnel.loc['Test',    next_step], funnel.loc['Test',    this_step] - funnel.loc['Test',    next_step]]
    ])
    
    # 跑卡方检验
    stat, p_val, dof, exp = chi2_contingency(contingency, correction=False)
    
    results.append({
        'Link': f"{this_step} -> {next_step}",
        'Control_Rate': funnel.loc['Control', next_step] / funnel.loc['Control', this_step],
        'Test_Rate':    funnel.loc['Test', next_step]    / funnel.loc['Test', this_step],
        'P-Value': p_val,
        'Significant': "Yes" if p_val < 0.05 else "No"
    })

# 输出结果
print("\n✅ 诊断完成 (Best Practice Analysis):")
display(pd.DataFrame(results).style.format({
    'Control_Rate': '{:.2%}', 
    'Test_Rate': '{:.2%}', 
    'P-Value': '{:.4f}'
}))

⚠️ 检测到 funnel 未定义，正在从 pd_funnel 重新构建...

✅ 诊断完成 (Best Practice Analysis):


/var/folders/35/q6rh83x91bzgf3gcsb_13f_80000gn/T/ipykernel_7919/641778114.py:15: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  funnel = df_funnel.groupby(['group','max_step'])['user_id'].nunique().unstack()


,Link,Control_Rate,Test_Rate,P-Value,Significant
0,View -> Cart,80.98%,86.30%,0.0170,Yes
1,Cart -> Pay,60.56%,71.09%,0.0007,Yes
2,Pay -> Success,49.47%,68.20%,0.0000,Yes


In [ ]:
# —————— 最佳步骤 ——————
# 定义漏斗步骤顺序
funnel_steps = ['View', 'Cart', 'Pay', 'Success']
results = []

# 循环每一个相邻的环节 (比如 View->Cart, Cart->Pay)
for i in range(len(funnel_steps)-1):
    this_step = funnel_steps[i]   # 当前环节 (分母)
    next_step = funnel_steps[i+1] # 下一环节 (分子)
    
    # 1. 构造该环节的 2x2 矩阵
    # 逻辑：[转化人数, 流失人数]
    contingency = np.array([
        [funnel.loc['Control', next_step], funnel.loc['Control', this_step] - funnel.loc['Control', next_step]],
        [funnel.loc['Test',    next_step], funnel.loc['Test',    this_step] - funnel.loc['Test',    next_step]]
    ])
    
    # 2. 跑卡方检验
    # correction=False (大样本不需要耶茨连续性修正)
    stat, p_val, dof, exp = chi2_contingency(contingency, correction=False)
    
    # 3. 记录结果
    results.append({
        'Link': f"{this_step} -> {next_step}",
        'Control_Rate': funnel.loc['Control', next_step] / funnel.loc['Control', this_step],
        'Test_Rate':    funnel.loc['Test', next_step]    / funnel.loc['Test', this_step],
        'P-Value': p_val,
        'Significant': "Yes" if p_val < 0.05 else "No"
    })

# 输出漂亮的诊断表
pd.DataFrame(results).style.format({
    'Control_Rate': '{:.2%}', 
    'Test_Rate': '{:.2%}', 
    'P-Value': '{:.4f}'
})
# ———————— 最佳步骤 ————————

In [ ]:
# --- Case 3: 你的代码 ---
# 1.  **Funnel Construction**: 计算各环节的**绝对人数**和**转化率**。
step_order = ['View', 'Cart', 'Pay', 'Success']
df_funnel['max_step'] = pd.Categorical(df_funnel['max_step'], categories=step_order, ordered=True)
df_funnel = df_funnel.sort_values('max_step')

funnel = df_funnel.groupby(['group','max_step'])['user_id'].nunique().unstack()
funnel = funnel.reset_index()
funnel

In [ ]:
funnel['cart/view'] = funnel['Cart']/funnel['View']
funnel['pay/cart'] = funnel['Pay']/funnel['Cart']
funnel['succes/pay'] = funnel['Success']/funnel['Pay']
total_views = funnel['View'].sum()
funnel['view%'] = funnel['View']/total_views

funnel

In [ ]:
# 2.  **Hypothesis Testing**: 使用 **Chi-Square Test (卡方检验)** 来判断两个漏斗分布是否有显著差异。
#     *   *提示*: `scipy.stats.chi2_contingency`
from scipy.stats import chisquare,chi2_contingency

# 观测值
observed = funnel['View'].values 
# 期望值 (假设大家平分)
expected = [total_views/2, total_views/2] 

stat, p_value = chisquare(observed, f_exp=expected)
print(f"SRM P-value: {p_value}") 
# 如果 p < 0.05, 说明流量分配严重不均，需要报警！
